In [2]:
import re
import requests
import pandas as pd
from bs4 import BeautifulSoup

HEADERS = {
    "User-Agent": "Research Script (contact: tz25@fsu.edu)"
}

RPO_TEXT_PATTERN = re.compile(
    r"remaining performance obligation",
    re.I
)

RPO_TAG_PATTERN = re.compile(
    r"RemainingPerformanceObligation",
    re.I
)


def sec_get(url):
    r = requests.get(url, headers=HEADERS, timeout=30)
    r.raise_for_status()
    return r.text


def filing_folder_url(html_url):
    return html_url.rsplit("/", 1)[0] + "/"


def clean_text(x):
    if x is None:
        return ""
    return " ".join(x.split())


def parse_xbrl_value(text):
    if text is None:
        return None

    x = text.replace(",", "").replace("$", "").replace("%", "").strip()

    try:
        return float(x)
    except ValueError:
        return text


def find_xbrl_instance_url(html_url):
    """
    从 filing folder 里找到真正的 XBRL instance XML。
    避免抓到 FilingSummary.xml、cal.xml、def.xml、lab.xml、pre.xml。
    """

    folder = filing_folder_url(html_url)
    index_html = sec_get(folder)
    soup = BeautifulSoup(index_html, "lxml")

    xml_candidates = []

    for a in soup.find_all("a"):
        href = a.get("href")

        if not href:
            continue

        file_name = href.split("/")[-1]

        if not file_name.lower().endswith(".xml"):
            continue

        lower_name = file_name.lower()

        # 排除明显不是 instance 的 XML
        if any(x in lower_name for x in [
            "filingsummary",
            "_cal",
            "_def",
            "_lab",
            "_pre",
            "cal.xml",
            "def.xml",
            "lab.xml",
            "pre.xml"
        ]):
            continue

        xml_candidates.append(folder + file_name)

    for xml_url in xml_candidates:
        try:
            xml_text = sec_get(xml_url)
            xml_soup = BeautifulSoup(xml_text, "lxml-xml")

            # 真正的 XBRL instance 通常有 xbrl root
            if xml_soup.find("xbrl"):
                return xml_url

        except Exception as e:
            print(f"Skip {xml_url}: {e}")
            continue

    return None


def clean_textblock_fact(value):
    soup = BeautifulSoup(value, "html.parser")
    return " ".join(soup.get_text(" ").split())


def build_context_map(xml_text):
    """
    读取 XBRL XML 里的所有 <context>，
    建立 contextRef -> context 信息 的字典。
    """

    soup = BeautifulSoup(xml_text, "lxml-xml")

    context_map = {}

    for ctx in soup.find_all("context"):

        ctx_id = ctx.get("id")

        if not ctx_id:
            continue

        # entity / CIK
        identifier_tag = ctx.find("identifier")

        entity_identifier = None
        entity_scheme = None

        if identifier_tag:
            entity_identifier = clean_text(identifier_tag.get_text())
            entity_scheme = identifier_tag.get("scheme")

        # period
        period = ctx.find("period")

        instant = None
        start_date = None
        end_date = None
        forever = False
        period_type = None

        if period:
            instant_tag = period.find("instant")
            start_tag = period.find("startDate")
            end_tag = period.find("endDate")
            forever_tag = period.find("forever")

            if instant_tag:
                instant = clean_text(instant_tag.get_text())
                period_type = "instant"

            elif start_tag and end_tag:
                start_date = clean_text(start_tag.get_text())
                end_date = clean_text(end_tag.get_text())
                period_type = "duration"

            elif forever_tag:
                forever = True
                period_type = "forever"

        # dimensions: segment + scenario
        dimensions = []

        for location_name in ["segment", "scenario"]:
            location = ctx.find(location_name)

            if not location:
                continue

            # explicitMember
            for member in location.find_all("explicitMember"):
                dimensions.append({
                    "location": location_name,
                    "member_type": "explicit",
                    "dimension": member.get("dimension"),
                    "member": clean_text(member.get_text())
                })

            # typedMember
            for member in location.find_all("typedMember"):
                dimensions.append({
                    "location": location_name,
                    "member_type": "typed",
                    "dimension": member.get("dimension"),
                    "member": clean_text(member.get_text())
                })

        context_map[ctx_id] = {
            "contextRef": ctx_id,
            "entity_identifier": entity_identifier,
            "entity_scheme": entity_scheme,
            "period_type": period_type,
            "instant": instant,
            "startDate": start_date,
            "endDate": end_date,
            "forever": forever,
            "dimensions": dimensions
        }

    return context_map


def build_unit_map(xml_text):
    """
    读取 XBRL XML 里的所有 <unit>，
    建立 unitRef -> unit 的字典。
    """

    soup = BeautifulSoup(xml_text, "lxml-xml")

    unit_map = {}

    for unit in soup.find_all("unit"):

        unit_id = unit.get("id")

        if not unit_id:
            continue

        measures = []

        for measure in unit.find_all("measure"):
            value = clean_text(measure.get_text())

            if value:
                measures.append(value)

        unit_map[unit_id] = " / ".join(measures)

    return unit_map


def extract_rpo_from_xbrl_xml(xml_url):
    """
    从 XBRL instance XML 提取：
    1. RPO numeric tags
    2. RPO-related TextBlock
    """

    xml = sec_get(xml_url)
    soup = BeautifulSoup(xml, "lxml-xml")

    # Context reference & Unit
    context_map = build_context_map(xml)
    unit_map = build_unit_map(xml)

    rows = []

    for tag in soup.find_all():
        tag_name = tag.name

        if not tag_name:
            continue

        if (
            "RemainingPerformanceObligation" in tag_name
            and any(x in tag_name for x in [
                "Axis",
                ".domain",
                "Domain",
            ])
        ):
            continue

        value_raw = clean_text(tag.get_text(" ", strip=True))

        if not value_raw:
            continue

        context_ref = tag.get("contextRef") or tag.get("contextref")
        context_info = context_map.get(context_ref, {})
        unit_ref = tag.get("unitRef") or tag.get("unitref")
        unit = unit_map.get(unit_ref)
        # decimals = tag.get("decimals")

        # 1. RPO numeric tags
        if RPO_TAG_PATTERN.search(tag_name):

            rows.append({
                "source": "xbrl_numeric",
                "xml_url": xml_url,
                "tag_name": tag_name,
                "value_raw": value_raw,

                # unit information
                "unit": unit,

                # context information
                "period_type": context_info.get("period_type"),
                "instant": context_info.get("instant"),
                "startDate": context_info.get("startDate"),
                "endDate": context_info.get("endDate"),
                "forever": context_info.get("forever"),
                "dimensions": context_info.get("dimensions")
                # "contextRef": context_info,
                # "value_num": parse_xbrl_value(value_raw),
                # "contextRef": context_ref,
                # "unitRef": unit_ref,
                # "decimals": decimals
            })

        # 2. RPO TextBlock
        tag_name_lower = tag_name.lower()

        if "textblock" in tag_name_lower:
            m = RPO_TEXT_PATTERN.search(value_raw)
            if m:
                print("matched keyword:", m.group(0))
                print("position:", m.start())

                rows.append({
                    "source": "xbrl_textblock",
                    "xml_url": xml_url,
                    "tag_name": tag_name,
                    "value_raw": value_raw,
                    # "value_num": None,
                    # "contextRef": context_info,
                    # "unitRef": unit_ref,
                    # "decimals": decimals

                    # unit information
                    "unit": unit,

                    # context information
                    "period_type": context_info.get("period_type"),
                    "instant": context_info.get("instant"),
                    "startDate": context_info.get("startDate"),
                    "endDate": context_info.get("endDate"),
                    "forever": context_info.get("forever"),
                    "dimensions": context_info.get("dimensions")

                })

    return rows




# def extract_rpo_paragraphs_from_html(html_url):
#     """
#     从 10-K HTML 正文中提取 RPO 相关段落。
#     """

#     html = sec_get(html_url)
#     soup = BeautifulSoup(html, "lxml")

#     paragraphs = []

#     for tag in soup.find_all(["p", "div", "td", "tr", "li"]):
#         text = clean_text(tag.get_text(" ", strip=True))

#         if len(text) < 40:
#             continue

#         if RPO_TEXT_PATTERN.search(text):
#             paragraphs.append(text)

#     # 去重，同时保持原顺序
#     paragraphs = list(dict.fromkeys(paragraphs))

#     return paragraphs


def process_filing(html_url):
    """
    输入 10-K HTML URL，
    输出一个 summary row + detailed RPO tags + paragraphs。
    """

    xml_url = find_xbrl_instance_url(html_url)

    if xml_url:
        rpo_tags = extract_rpo_from_xbrl_xml(xml_url)
    else:
        rpo_tags = []

    # paragraphs = extract_rpo_paragraphs_from_html(html_url)

    row = {
        "html_url": html_url,
        "xml_url": xml_url,
        "num_rpo_tags": len(rpo_tags)
        # "rpo_paragraphs": "\n\n---\n\n".join(paragraphs)
    }

    for i, item in enumerate(rpo_tags, 1):
        row[f"Tag_{i}_source"] = item["source"]
        row[f"Tag_{i}_name"] = item["tag_name"]
        row[f"Tag_{i}_value_raw"] = item["value_raw"]
        # row[f"Tag_{i}_value_num"] = item["value_num"]
        row[f"Tag_{i}_unit"] = item["unit"]
        # row[f"Tag_{i}_contextRef"] = item["contextRef"]
        # row[f"Tag_{i}_unitRef"] = item["unitRef"]
        # row[f"Tag_{i}_decimals"] = item["decimals"]
        row[f"Tag_{i}_period_type"] = item["period_type"]
        row[f"Tag_{i}_instant"] = item["instant"]
        row[f"Tag_{i}_startDate"] = item["startDate"]
        row[f"Tag_{i}_endDate"] = item["endDate"]
        row[f"Tag_{i}_dimensions"] = item["dimensions"]

    return row, rpo_tags



In [3]:
xml = "https://www.sec.gov/Archives/edgar/data/1341439/000156459019023119/orcl-20190531.xml"

In [4]:
extract_rpo_from_xbrl_xml(xml)

matched keyword: Remaining Performance Obligation
position: 164292
matched keyword: Remaining Performance Obligation
position: 23377


[{'source': 'xbrl_textblock',
  'xml_url': 'https://www.sec.gov/Archives/edgar/data/1341439/000156459019023119/orcl-20190531.xml',
  'tag_name': 'OrganizationConsolidationAndPresentationOfFinancialStatementsDisclosureAndSignificantAccountingPoliciesTextBlock',
  'value_raw': '<div align="left"> <table border="0" cellspacing="0" cellpadding="0" style="border-collapse:collapse; width:100%;"> <tr> <td valign="top" style="width:5.24%;white-space:nowrap"> <p style="text-align:justify;margin-top:12pt;margin-bottom:0pt;font-weight:bold;font-family:Calibri;font-size:10pt;font-style:normal;text-transform:none;font-variant: normal;"><font style="font-weight:bold;font-family:Calibri;font-size:10pt;font-style:normal;text-transform:none;font-variant: normal;">1.</font></p></td> <td valign="top"> <p style="text-align:justify;margin-top:12pt;margin-bottom:0pt;font-weight:bold;font-style:normal;text-transform:none;font-variant: normal;font-family:Calibri;font-size:10pt;"><a name="N1_ORGANIZATION_SIGNI

In [5]:
df = pd.read_excel("company_2015_2025.xlsx")

rows = []

# 每处理 100 个 filing，自动保存一次
SAVE_EVERY = 100

for i, (_, filing) in enumerate(df.iterrows(), 1):

    url = filing["html"]

    print(f"Processing {i}/{len(df)}: {url}")

    try:
        # 检查 URL 是否为空
        if pd.isna(url) or not str(url).strip():
            raise ValueError("Missing HTML URL")

        row, rpo_tags = process_filing(url)

        row["error"] = None

    except Exception as e:
        print(f"Error: {e}")

        row = {
            "html_url": url,
            "xml_url": None,
            "num_rpo_tags": 0,
            "num_rpo_paragraphs": 0,
            "rpo_paragraphs": None,
            "error": str(e)
        }

    # 添加公司信息
    row["CIK"] = filing["CIK"]
    row["Company"] = filing["Company"]
    row["FiscalYear"] = filing["fiscalYear"]

    rows.append(row)

    # 每 100 个 filing 自动保存一次
    if i % SAVE_EVERY == 0:

        df_temp = pd.DataFrame(rows)

        df_temp.to_excel(
            "rpo_new_xbrl_output_checkpoint.xlsx",
            index=False
        )

        df_temp.to_csv(
            "rpo_new_xbrl_output_checkpoint.csv",
            index=False,
            encoding="utf-8-sig"
        )

        print(f"Checkpoint saved: {i}/{len(df)}")


# 最终输出
df_out = pd.DataFrame(rows)

df_out.to_excel(
    "rpo_xbrl_output.xlsx",
    index=False
)

df_out.to_csv(
    "rpo_xbrl_output.csv",
    index=False,
    encoding="utf-8-sig"
)

print("Done.")
print(f"Total filings processed: {len(df_out)}")
print(f"Output rows: {len(df_out)}")

Processing 1/44: https://www.sec.gov/Archives/edgar/data/1341439/000119312515235239/d920711d10k.htm
Processing 2/44: https://www.sec.gov/Archives/edgar/data/1341439/000119312516628942/d203801d10k.htm
Processing 3/44: https://www.sec.gov/Archives/edgar/data/1341439/000119312517214833/d385998d10k.htm
Processing 4/44: https://www.sec.gov/Archives/edgar/data/1341439/000119312518201034/d568983d10k.htm
Processing 5/44: https://www.sec.gov/Archives/edgar/data/1341439/000156459019023119/orcl-10k_20190531.htm
matched keyword: Remaining Performance Obligation
position: 164292
matched keyword: Remaining Performance Obligation
position: 23377
Processing 6/44: https://www.sec.gov/Archives/edgar/data/1341439/000156459020030125/orcl-10k_20200531.htm
matched keyword: Remaining Performance Obligation
position: 35311
matched keyword: Remaining Performance Obligation
position: 27496
Processing 7/44: https://www.sec.gov/Archives/edgar/data/1341439/000156459021033616/orcl-10k_20210531.htm
matched keyword: 

C:\Users\tz25\AppData\Local\Temp\ipykernel_87400\303808934.py:64: UserWarning: Cell contents too long (270973), truncated to 32767 characters
  df_out.to_excel(
C:\Users\tz25\AppData\Local\Temp\ipykernel_87400\303808934.py:64: UserWarning: Cell contents too long (150233), truncated to 32767 characters
  df_out.to_excel(
C:\Users\tz25\AppData\Local\Temp\ipykernel_87400\303808934.py:64: UserWarning: Cell contents too long (49768), truncated to 32767 characters
  df_out.to_excel(
C:\Users\tz25\AppData\Local\Temp\ipykernel_87400\303808934.py:64: UserWarning: Cell contents too long (52112), truncated to 32767 characters
  df_out.to_excel(
C:\Users\tz25\AppData\Local\Temp\ipykernel_87400\303808934.py:64: UserWarning: Cell contents too long (52341), truncated to 32767 characters
  df_out.to_excel(
C:\Users\tz25\AppData\Local\Temp\ipykernel_87400\303808934.py:64: UserWarning: Cell contents too long (533247), truncated to 32767 characters
  df_out.to_excel(
C:\Users\tz25\AppData\Local\Temp\ipyk

In [ ]:
import requests

headers = {
    "User-Agent": "Research Script (contact: tz25@fsu.edu)"
}

url = "https://www.sec.gov/Archives/edgar/data/1318605/000156459019003165/tsla-20181231.xml"

xml = requests.get(url, headers=headers).text

print(len(xml))


7034413


In [ ]:
with open("tesla.xml", "w", encoding="utf-8") as f:
    f.write(xml)